# Adaptive transport networks: where we are

**Top goal.** Find out how a pattern growing in space and the network that feeds it shape each other. The concrete question: *does the kind of pattern (static spots vs dividing spots) decide whether the network ends as a **tree** (no loops) or keeps **loops**?*

| Step | Question | Status |
|:--|:--|:--|
| A | Steady load: what decides tree vs loops? | done, tested |
| B | Moving load: are loops kept even where a steady load gives a tree? | done, tested |
| C | Couple to Gray-Scott: static spots give trees? dividing spots give loops? | **next, not built** |

**Contents:** 1 the model, 2 explorer (Steps A and B), 3 what to try, 4 headline results, 5 the γ scan, 6 the chemistry, 7 Step C.

**How to use:** run top to bottom. Needs `ipywidgets`. Sliders change the setup, **Run** simulates, and the *time* and *cutoff* sliders then scrub through the result instantly.

In [ ]:
import os
import time
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as w
from IPython.display import display

from pattern_formation.networks import (
    grid_edges, count_loops, relax, relax_fluctuating, draw_network,
)
from pattern_formation.reaction_diffusion import gray_scott, render_rd

# Tests run this notebook as plain Python; keep everything tiny there.
FAST = "PYTEST_CURRENT_TEST" in os.environ


def layout(n):
    """Nodes of an n x n grid: the centre is the source, every other node a sink."""
    N = n * n
    centre = N // 2
    return N, centre, [i for i in range(N) if i != centre]


def run_history(n, gamma, steps, nu=1.0, load="steady", k=5, seed=0, frames=40):
    """Adapt a network from C = 1 and keep a snapshot of C about `frames` times."""
    N, centre, sinks = layout(n)
    chunk = max(1, steps // frames)
    rng = np.random.default_rng(seed)
    edges = grid_edges(n)
    C = np.ones(len(edges))
    times, snaps = [0], [C]
    for _ in range(steps // chunk):
        if load == "steady":
            _, C = relax(n, centre, sinks, nu=nu, gamma=gamma, steps=chunk, C0=C)
        else:
            _, C = relax_fluctuating(n, centre, sinks, k=k, nu=nu, gamma=gamma,
                                     steps=chunk, rng=rng, C0=C)
        times.append(times[-1] + chunk)
        snaps.append(C)
    return dict(n=n, gamma=gamma, nu=nu, load=load, edges=edges, times=times, snaps=snaps)


def leaf_conductances(edges, C, N, cutoff):
    """C of the links that end a branch (one endpoint has exactly one kept link)."""
    keep = C > cutoff
    leaf = np.bincount(edges[keep].ravel(), minlength=N) == 1
    return C[keep & (leaf[edges[:, 0]] | leaf[edges[:, 1]])]


## 1. The model

Pixels of a grid are **nodes**; neighbouring pixels are joined by **links**. Fluid enters at one **source** (centre) and leaves at **sinks** (every other node, one unit each).

| symbol | meaning |
|:--|:--|
| $C_{ij}$ | conductance of the link between nodes $i$ and $j$ (how wide the pipe is) |
| $p_i$ | pressure at node $i$ |
| $s_i$ | net inflow at node $i$: positive at the source, $-1$ at a sink, $\sum_i s_i = 0$ |
| $Q_{ij}$ | flow along the link |

**Step 1: flow follows pressure.** For the current $C$, solve one linear system (flow in = flow out at every node):

$$\sum_j C_{ij}\,(p_i - p_j) = s_i, \qquad Q_{ij} = C_{ij}\,(p_i - p_j).$$

**Step 2: pipes adapt to the flow.**

$$\frac{dC}{dt} \;=\; \underbrace{\frac{Q^2}{C}}_{\text{busy links widen}} \;-\; \underbrace{\nu\,C^{\gamma}}_{\text{upkeep cost}}$$

Repeat 1 and 2 until nothing changes. $\nu$ scales the upkeep; $\gamma$ says how fast upkeep grows with width.

**Prediction 1: a settled link's width.** At rest growth equals upkeep, $Q^2/C = \nu C^\gamma$, so

$$C^* = \left(\frac{Q^2}{\nu}\right)^{1/(\gamma+1)}.$$

A branch tip drains one unit ($Q = 1$), so its width should be $\nu^{-1/(\gamma+1)}$, which is $1$ for $\nu = 1$.

**Prediction 2: tree or loops.** At rest, a link carrying flow $Q$ costs $\nu C^{*\gamma} \propto Q^{\,2\gamma/(\gamma+1)}$. If that exponent is below 1 (that is, $\gamma < 1$), one wide link carrying merged flow is cheaper than two thin ones, so flows merge and a **tree** results. Above 1 ($\gamma > 1$), splitting is cheaper and **loops** survive. The exponent equals 1 exactly at $\gamma = 1$.

**Moving load (Step B).** Instead of one fixed set of sinks, each step averages over random sets of $k$ active sinks:

$$\frac{dC}{dt} = \frac{\langle Q^2\rangle}{C} - \nu\,C^{\gamma}.$$

The idea: a link idle under one draw is busy under another, so it is not starved away.

**Counting loops.** Keep links with $C >$ cutoff, where cutoff $= 10^{x}\,C_{\max}$. With $E$ kept links, $N$ nodes and $K$ connected pieces,

$$\text{loops} = E - N + K.$$

A tree has $E = N-1$, $K = 1$, so 0 loops. The full $n \times n$ grid has $(n-1)^2$ loops, the most possible.

## 2. Explorer (Steps A and B)

Left: the network at the chosen time (line width $\propto \sqrt{C}$, so thin links stay visible). Right: loops over time; the dashed line is the most possible, $(n-1)^2$.

In [ ]:
def show(run, i, cut_exp):
    n, edges = run["n"], run["edges"]
    N = n * n
    C = run["snaps"][i]
    cut = 10.0 ** cut_exp * C.max()
    loops = [count_loops(edges, c, N, 10.0 ** cut_exp * c.max()) for c in run["snaps"]]
    fig, (a, b) = plt.subplots(1, 2, figsize=(10, 4.4), gridspec_kw={"width_ratios": [1, 1.1]})
    draw_network(a, n, edges, np.sqrt(C), cutoff=np.sqrt(cut), width=5)
    a.set_title(f"after {run['times'][i]} steps")
    b.plot(run["times"], loops, "o-", ms=3)
    b.axhline((n - 1) ** 2, ls="--", c="gray")
    b.axvline(run["times"][i], c="C3")
    b.set_xlabel("adaptation steps")
    b.set_ylabel("independent loops")
    b.set_title("is it still changing? (flat = settled)")
    plt.tight_layout()
    plt.show()
    kept = int((C > cut).sum())
    print(f"loops: {loops[i]}   links kept: {kept} of {len(C)}   "
          f"(a tree on {N} nodes has {N - 1})")
    leaf = leaf_conductances(edges, C, N, cut)
    if run["load"] == "steady" and len(leaf):
        pred = run["nu"] ** (-1 / (run["gamma"] + 1))
        print(f"branch tips: median C = {np.median(leaf):.3f}   predicted {pred:.3f}")


def slider(kind, **kw):
    return kind(style={"description_width": "9em"}, layout=w.Layout(width="26em"), **kw)


D = dict(n=5, steps=200) if FAST else dict(n=9, steps=4000)
load = w.ToggleButtons(options=[("steady (Step A)", "steady"), ("moving (Step B)", "moving")],
                       value="steady", description="load")
gamma = slider(w.FloatSlider, value=0.5, min=0.2, max=2.0, step=0.05, description="γ (upkeep exponent)")
nu = slider(w.FloatSlider, value=1.0, min=0.2, max=3.0, step=0.1, description="ν (upkeep scale)")
n_w = slider(w.IntSlider, value=D["n"], min=5, max=13, description="grid size n")
steps_w = slider(w.IntSlider, value=D["steps"], min=200, max=10000, step=200, description="steps")
k_w = slider(w.IntSlider, value=5, min=1, max=20, description="k (moving only)")
seed_w = slider(w.IntSlider, value=0, min=0, max=9, description="seed (moving only)")
run_btn = w.Button(description="Run", button_style="primary")
status = w.HTML()
frame = slider(w.IntSlider, value=0, min=0, max=40, description="time", continuous_update=False)
cut_exp = slider(w.FloatSlider, value=-3, min=-5, max=-1, step=0.25,
                 description="cutoff exponent x", continuous_update=False)
out = w.Output()
STATE = {}


def redraw(*_):
    run = STATE.get("run")
    if run is None:
        return
    with out:
        out.clear_output(wait=True)
        show(run, min(frame.value, len(run["times"]) - 1), cut_exp.value)


def on_run(_):
    run_btn.disabled = True
    status.value = "running..."
    t0 = time.time()
    STATE["run"] = run_history(n_w.value, gamma.value, steps_w.value, nu.value,
                               load.value, k_w.value, seed_w.value)
    last = len(STATE["run"]["times"]) - 1
    frame.max = last
    status.value = f"done in {time.time() - t0:.1f} s"
    run_btn.disabled = False
    if frame.value == last:
        redraw()
    else:
        frame.value = last  # triggers redraw


run_btn.on_click(on_run)
frame.observe(redraw, "value")
cut_exp.observe(redraw, "value")
display(w.VBox([load, gamma, nu, n_w, steps_w, k_w, seed_w,
                w.HBox([run_btn, status]), frame, cut_exp, out]))
on_run(None)


## 3. What to try

Predict first, then run.

1. **Steady, $\gamma = 0.5$.** Prediction 2 says a tree. Does the curve reach 0 and stay flat? Does the loop count depend on the cutoff once it has settled?
2. **Steady, $\gamma = 1.5$.** Prediction 2 says loops survive. How many of the $(n-1)^2$ possible?
3. **Branch tips.** Under steady load, change $\nu$ and $\gamma$. Does the printed median tip width follow Prediction 1?
4. **Small $\gamma$ (0.3).** Settling is slow. Does 4000 steps leave loops behind? What does more time do?
5. **Moving load, $\gamma = 0.5$.** Steady load gave a tree here. What happens now? Does it change with $k$, the seed, or the cutoff?

## 4. Headline results against expectations

Three runs on an $11 \times 11$ grid (source at the centre). Loops are counted at three cutoffs, because for the moving load the count depends on it. Press the button to reproduce (about a minute).

In [ ]:
def headline(n=11, steady_steps=8000, moving_steps=4000, seeds=(0, 1, 2)):
    N, centre, sinks = layout(n)
    cuts = (1e-4, 1e-3, 1e-2)

    def loops(e, C):
        return [count_loops(e, C, N, c * C.max()) for c in cuts]

    rows = []
    for g in (0.5, 1.5):
        e, C = relax(n, centre, sinks, gamma=g, steps=steady_steps)
        rows.append((f"steady, gamma={g}", loops(e, C)))
    for sd in seeds:
        e, C = relax_fluctuating(n, centre, sinks, gamma=0.5, steps=moving_steps,
                                 rng=np.random.default_rng(sd))
        rows.append((f"moving, gamma=0.5, seed {sd}", loops(e, C)))
    return N, rows


def print_headline(n, steady_steps, moving_steps, seeds):
    N, rows = headline(n, steady_steps, moving_steps, seeds)
    print(f"{'run':30s} loops at cutoff 1e-4 / 1e-3 / 1e-2 of Cmax")
    for name, l in rows:
        print(f"{name:30s} {l[0]:4d} {l[1]:4d} {l[2]:4d}")
    print(f"\nExpected: steady gamma=0.5 -> 0 (a tree has N-1 = {N - 1} links); "
          f"steady gamma=1.5 -> many (at most (n-1)^2 = {(n - 1) ** 2}); "
          f"moving gamma=0.5 -> clearly above 0.")


head_out = w.Output()
head_btn = w.Button(description="Run headline experiments (~1 min)", button_style="primary")


def on_head(_):
    head_btn.disabled = True
    with head_out:
        head_out.clear_output()
        print("running...")
        print_headline(11, 8000, 4000, (0, 1, 2))
    head_btn.disabled = False


head_btn.on_click(on_head)
display(w.VBox([head_btn, head_out]))
if FAST:  # exercise the code on a tiny grid
    print_headline(5, 100, 100, (0,))


## 5. The $\gamma$ scan

Prediction 2 puts the switch from tree to loops at $\gamma = 1$, where the cost exponent $2\gamma/(\gamma+1)$ equals 1. Steady load, $9 \times 9$ grid. Very small $\gamma$ settles slowly, so the leftmost points may not have finished.

In [ ]:
def scan_gamma(n=9, steps=5000, gammas=(0.3, 0.5, 0.7, 0.9, 1.0, 1.1, 1.3, 1.5, 2.0)):
    N, centre, sinks = layout(n)
    loops = []
    for g in gammas:
        e, C = relax(n, centre, sinks, gamma=g, steps=steps)
        loops.append(count_loops(e, C, N, 1e-3 * C.max()))
    return np.array(gammas), np.array(loops)


def plot_scan(n, steps, gammas):
    g, loops = scan_gamma(n, steps, gammas)
    fig, ax = plt.subplots(figsize=(6, 3.6))
    ax.plot(g, loops, "o-")
    ax.axhline((n - 1) ** 2, ls="--", c="gray", label="most possible")
    ax.axvline(1.0, c="C3", label="predicted switch")
    ax.set_xlabel("gamma")
    ax.set_ylabel("independent loops")
    ax.legend()
    plt.tight_layout()
    plt.show()


scan_out = w.Output()
scan_btn = w.Button(description="Run γ scan (~40 s)", button_style="primary")


def on_scan(_):
    scan_btn.disabled = True
    with scan_out:
        scan_out.clear_output()
        print("running...")
        plot_scan(9, 5000, (0.3, 0.5, 0.7, 0.9, 1.0, 1.1, 1.3, 1.5, 2.0))
    scan_btn.disabled = False


scan_btn.on_click(on_scan)
display(w.VBox([scan_btn, scan_out]))
if FAST:
    plot_scan(5, 100, (0.5, 1.5))


## 6. The other ingredient (already built): Gray-Scott

Two chemicals on a periodic grid. $A$ is fed in at rate $f$; $B$ eats $A$, makes more $B$, and dies at rate $k+f$:

$$\dot A = d_A \nabla^2 A - A B^2 + f(1-A), \qquad \dot B = d_B \nabla^2 B + A B^2 - (k+f)\,B.$$

The term $AB^2$ is the chemistry's **consumption** of $A$. In Step C it becomes the sinks of the network. The sliders start at the *dividing spots* (mitosis) setting, $f = 0.0367$, $k = 0.0649$.

In [ ]:
f_w = slider(w.FloatSlider, value=0.0367, min=0.01, max=0.07, step=0.0005,
             readout_format=".4f", description="f (feed)", continuous_update=False)
k_chem = slider(w.FloatSlider, value=0.0649, min=0.04, max=0.07, step=0.0005,
                readout_format=".4f", description="k (kill)", continuous_update=False)
chem_steps = slider(w.IntSlider, value=50 if FAST else 6000, min=50, max=20000, step=50,
                    description="steps", continuous_update=False)
chem_out = w.Output()


def draw_chem(*_):
    size = 32 if FAST else 128
    _, B = gray_scott((size, size), f=f_w.value, k=k_chem.value, steps=chem_steps.value,
                      rng=np.random.default_rng(0), init="spots")
    with chem_out:
        chem_out.clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(4.5, 4.5))
        ax.imshow(render_rd(B))
        ax.axis("off")
        plt.show()


for s in (f_w, k_chem, chem_steps):
    s.observe(draw_chem, "value")
display(w.VBox([f_w, k_chem, chem_steps, chem_out]))
draw_chem()


## 7. Step C (next, not built): couple the two

The Gray-Scott grid *is* the network: pixels are nodes, neighbouring pixels are links with conductance $C_{ij}$. Alternate **many chemistry steps, then one adaptation step**, with two changes to the chemistry:

- **Spreading follows the links.** Replace $d_A \nabla^2 A$ by $\nabla\cdot(C\,\nabla A)$, not $C\,\nabla^2 A$, so the total amount of $A$ is conserved.
- **Consumption becomes the sinks.** The $AB^2$ consumption at each node becomes that node's $s_i$ in Step 1 of the model.

**The question.** Static spots: do they give a tree? Dividing spots (mitosis, $f=0.0367$, $k=0.0649$): do they give loops?

**Open design choices (not decided yet).**

- Where the source sits.
- How consumption is scaled so that $\sum_i s_i = 0$.
- How many chemistry steps per adaptation step.
- The chemistry uses a 9-point stencil, while the network has 4 links per node. How to reconcile them.

The "Later" list is in handoff 01 (a local file, not in the repository).